<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<h2 style="font-family: SimSun, 宋体, serif; font-size: 15pt; line-height: 1.5; margin: 0; text-align: justify;">Task 2：数据诊断与技术指标</h2>
</div>

<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<h3 style="font-family: SimSun, 宋体, serif; font-size: 12pt; line-height: 1.5; margin: 0; text-align: justify;">1. 数据基础诊断分析</h3>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
对数据进行全面的诊断分析，包括以下内容：
</p>
<ul style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>检查缺失值：统计每个字段的缺失值数量</li>
<li>数据完整性分析：检查数据的时间序列连续性</li>
<li>描述性统计量：计算均值、标准差、最小值、最大值、中位数、分位数等</li>
<li>数据分布检查：观察价格和成交量的分布特征</li>
<li>异常值检测：识别可能的异常数据点</li>
</ul>
</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimSun']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据
df = pd.read_csv('../data/task1_daily_data.csv')
df = df.sort_values('trade_date').reset_index(drop=True)

# 将交易日期转换为日期格式
df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')

print('='*80)
print('数据基础诊断分析')
print('='*80)

# 1. 检查缺失值
print('\n【1. 缺失值检查】')
print(df.isna().sum())

# 2. 数据基本信息
print('\n【2. 数据基本信息】')
print(f'数据行数：{len(df)}')
print(f'数据列数：{len(df.columns)}')
print(f'起始日期：{df["trade_date"].min()}')
print(f'结束日期：{df["trade_date"].max()}')
print(f'时间跨度：{(df["trade_date"].max() - df["trade_date"].min()).days} 天')

# 3. 描述性统计量
print('\n【3. 描述性统计量】')
display(df.describe())

# 4. 价格涨跌幅分析
df['price_change'] = df['close'].pct_change() * 100
print('\n【4. 价格涨跌幅统计】')
print(f'平均日涨跌幅：{df["price_change"].mean():.4f}%')
print(f'涨跌幅标准差：{df["price_change"].std():.4f}%')
print(f'最大日涨幅：{df["price_change"].max():.4f}%')
print(f'最大日跌幅：{df["price_change"].min():.4f}%')

# 5. 绘制价格走势图
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 收盘价走势
axes[0].plot(df['trade_date'], df['close'], label='收盘价', color='blue', linewidth=1.5)
axes[0].set_title('收盘价走势', fontsize=12)
axes[0].set_xlabel('日期', fontsize=10)
axes[0].set_ylabel('价格', fontsize=10)
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# 成交量走势
axes[1].bar(df['trade_date'], df['vol'], label='成交量', color='green', alpha=0.6)
axes[1].set_title('成交量走势', fontsize=12)
axes[1].set_xlabel('日期', fontsize=10)
axes[1].set_ylabel('成交量', fontsize=10)
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

# 涨跌幅分布
axes[2].hist(df['price_change'].dropna(), bins=50, color='orange', alpha=0.7, edgecolor='black')
axes[2].set_title('日涨跌幅分布', fontsize=12)
axes[2].set_xlabel('涨跌幅 (%)', fontsize=10)
axes[2].set_ylabel('频数', fontsize=10)
axes[2].axvline(x=0, color='red', linestyle='--', linewidth=1)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<h3 style="font-family: SimSun, 宋体, serif; font-size: 12pt; line-height: 1.5; margin: 0; text-align: justify;">2. 技术指标介绍</h3>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
本部分将详细介绍三个经典技术指标的计算方法和作用：RSI、MACD、布林带（Bollinger Bands）。
</p>
</div>

<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<h4 style="font-family: SimSun, 宋体, serif; font-size: 11pt; line-height: 1.5; margin: 0; text-align: justify;">2.1 RSI（相对强弱指标）</h4>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>定义：</strong>RSI（Relative Strength Index，相对强弱指标）是由 Welles Wilder 于 1978 年提出的动量指标，用于衡量价格变动的速度和变化。
</p>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>计算方法：</strong>
</p>
<ol style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>计算每日价格变化：Δ = 当日收盘价 - 前一日收盘价</li>
<li>计算平均涨幅（Gain）和平均跌幅（Loss）：使用 N 日（通常为 14 日）滚动平均</li>
<li>计算相对强弱 RS = 平均涨幅 / 平均跌幅</li>
<li>计算 RSI = 100 - 100 / (1 + RS)</li>
</ol>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>作用：</strong>
</p>
<ul style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>超买超卖信号：RSI &gt; 70 视为超买，可能回调；RSI &lt; 30 视为超卖，可能反弹</li>
<li>背离信号：价格创新高但 RSI 未创新高（顶背离），或价格创新低但 RSI 未创新低（底背离）</li>
<li>趋势判断：RSI 在 50 以上表明上涨趋势，50 以下表明下跌趋势</li>
</ul>
</div>

<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<h4 style="font-family: SimSun, 宋体, serif; font-size: 11pt; line-height: 1.5; margin: 0; text-align: justify;">2.2 MACD（指数平滑异同移动平均线）</h4>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>定义：</strong>MACD（Moving Average Convergence Divergence）是由 Gerald Appel 于 1979 年提出的趋势跟踪指标，通过计算两条移动平均线的差离值来判断趋势。
</p>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>计算方法：</strong>
</p>
<ol style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>计算短期 EMA（通常为 12 日指数移动平均）</li>
<li>计算长期 EMA（通常为 26 日指数移动平均）</li>
<li>计算 DIF（差离值）= EMA(12) - EMA(26)</li>
<li>计算 DEA（信号线）= EMA(DIF, 9)，即 DIF 的 9 日指数移动平均</li>
<li>计算 MACD 柱状图 = 2 × (DIF - DEA)</li>
</ol>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>作用：</strong>
</p>
<ul style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>金叉信号：DIF 从下向上突破 DEA，为买入信号</li>
<li>死叉信号：DIF 从上向下突破 DEA，为卖出信号</li>
<li>趋势判断：DIF 和 DEA 均在零轴上方，表明上涨趋势；均在零轴下方，表明下跌趋势</li>
<li>柱状图变化：柱状图由负转正或由正转负，表明动量的变化</li>
</ul>
</div>

<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<h4 style="font-family: SimSun, 宋体, serif; font-size: 11pt; line-height: 1.5; margin: 0; text-align: justify;">2.3 布林带（Bollinger Bands）</h4>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>定义：</strong>布林带由 John Bollinger 于 1980 年代提出，是一种基于统计标准差的 volatility 指标，由三条线组成：中轨、上轨和下轨。
</p>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>计算方法：</strong>
</p>
<ol style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>计算中轨：N 日简单移动平均线（通常 N = 20）</li>
<li>计算标准差：N 日收盘价的标准差</li>
<li>计算上轨 = 中轨 + 2 × 标准差</li>
<li>计算下轨 = 中轨 - 2 × 标准差</li>
</ol>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>作用：</strong>
</p>
<ul style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>波动区间：布林带宽度表示市场波动大小，带宽收窄表示波动减小，带宽扩大表示波动增大</li>
<li>支撑阻力：上轨通常作为阻力位，下轨通常作为支撑位</li>
<li>超买超卖：价格触及上轨可能超买，触及下轨可能超卖</li>
<li>趋势判断：价格在中轨上方运行表明上涨趋势，在中轨下方运行表明下跌趋势</li>
</ul>
</div>

<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<h3 style="font-family: SimSun, 宋体, serif; font-size: 12pt; line-height: 1.5; margin: 0; text-align: justify;">3. Python 编程实现技术指标计算与可视化</h3>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
下面使用 Python 编程加载数据，计算 RSI、MACD、布林带三个指标，并绘制可视化图形。
</p>
</div>

In [ ]:
# 计算技术指标
close = df['close']

# 3.1 计算 RSI（14日）
delta = close.diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
df['RSI'] = 100 - 100 / (1 + gain / loss)

# 3.2 计算 MACD
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
df['DIF'] = ema12 - ema26
df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
df['MACD'] = 2 * (df['DIF'] - df['DEA'])

# 3.3 计算布林带（20日，2倍标准差）
df['BOLL_MID'] = close.rolling(20).mean()
df['BOLL_STD'] = close.rolling(20).std()
df['BOLL_UPPER'] = df['BOLL_MID'] + 2 * df['BOLL_STD']
df['BOLL_LOWER'] = df['BOLL_MID'] - 2 * df['BOLL_STD']

# 绘制技术指标图表
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(4, 1, height_ratios=[3, 2, 2, 2], hspace=0.3)

# 子图 1：布林带
ax1 = fig.add_subplot(gs[0])
ax1.plot(df['trade_date'], df['close'], label='收盘价', color='black', linewidth=1.5)
ax1.plot(df['trade_date'], df['BOLL_UPPER'], label='上轨', color='red', linestyle='--', linewidth=1)
ax1.plot(df['trade_date'], df['BOLL_MID'], label='中轨', color='blue', linewidth=1.2)
ax1.plot(df['trade_date'], df['BOLL_LOWER'], label='下轨', color='green', linestyle='--', linewidth=1)
ax1.fill_between(df['trade_date'], df['BOLL_LOWER'], df['BOLL_UPPER'], alpha=0.1, color='gray')
ax1.set_title('布林带（Bollinger Bands）', fontsize=14, fontweight='bold')
ax1.set_ylabel('价格', fontsize=12)
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# 子图 2：RSI
ax2 = fig.add_subplot(gs[1])
ax2.plot(df['trade_date'], df['RSI'], label='RSI', color='purple', linewidth=1.5)
ax2.axhline(y=70, color='red', linestyle='--', linewidth=1, label='超买线 (70)')
ax2.axhline(y=30, color='green', linestyle='--', linewidth=1, label='超卖线 (30)')
ax2.axhline(y=50, color='gray', linestyle=':', linewidth=1)
ax2.set_title('RSI（相对强弱指标）', fontsize=14, fontweight='bold')
ax2.set_ylabel('RSI', fontsize=12)
ax2.set_ylim([0, 100])
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

# 子图 3：MACD
ax3 = fig.add_subplot(gs[2])
ax3.plot(df['trade_date'], df['DIF'], label='DIF', color='blue', linewidth=1.5)
ax3.plot(df['trade_date'], df['DEA'], label='DEA', color='orange', linewidth=1.5)
colors = ['red' if x &gt;= 0 else 'green' for x in df['MACD']]
ax3.bar(df['trade_date'], df['MACD'], label='MACD 柱', color=colors, alpha=0.6, width=1)
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax3.set_title('MACD（指数平滑异同移动平均线）', fontsize=14, fontweight='bold')
ax3.set_ylabel('MACD', fontsize=12)
ax3.legend(loc='best')
ax3.grid(True, alpha=0.3)

# 子图 4：成交量
ax4 = fig.add_subplot(gs[3])
ax4.bar(df['trade_date'], df['vol'], label='成交量', color='teal', alpha=0.7)
ax4.set_title('成交量', fontsize=14, fontweight='bold')
ax4.set_xlabel('日期', fontsize=12)
ax4.set_ylabel('成交量', fontsize=12)
ax4.legend(loc='best')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<h3 style="font-family: SimSun, 宋体, serif; font-size: 12pt; line-height: 1.5; margin: 0; text-align: justify;">4. 扩展指标：OBV（能量潮）</h3>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>定义：</strong>OBV（On Balance Volume，能量潮）是由 Joseph Granville 于 1963 年提出的技术分析指标，将成交量与价格变动相结合，用于衡量资金流向。
</p>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>计算方法：</strong>
</p>
<ol style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>当日收盘价 &gt; 前一日收盘价：OBV = 前一日 OBV + 当日成交量</li>
<li>当日收盘价 &lt; 前一日收盘价：OBV = 前一日 OBV - 当日成交量</li>
<li>当日收盘价 = 前一日收盘价：OBV = 前一日 OBV</li>
</ol>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<strong>作用：</strong>
</p>
<ul style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>确认趋势：OBV 与价格同向变动时，趋势得到确认</li>
<li>背离信号：价格创新高但 OBV 未创新高（顶背离），或价格创新低但 OBV 未创新低（底背离），预示趋势可能反转</li>
<li>资金流向：OBV 上升表明资金流入，OBV 下降表明资金流出</li>
<li>突破确认：价格突破时 OBV 也同步突破，确认突破有效</li>
</ul>
</div>

In [ ]:
# 计算 OBV
direction = close.diff().apply(lambda x: 1 if x &gt; 0 else (-1 if x &lt; 0 else 0))
df['OBV'] = (direction * df['vol']).fillna(0).cumsum()

# 绘制 OBV 图表
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [2, 1.5]})

# 价格走势
ax1.plot(df['trade_date'], df['close'], label='收盘价', color='blue', linewidth=1.5)
ax1.set_title('收盘价走势', fontsize=14, fontweight='bold')
ax1.set_ylabel('价格', fontsize=12)
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# OBV
ax2.plot(df['trade_date'], df['OBV'], label='OBV', color='darkgreen', linewidth=1.5)
ax2.set_title('OBV（能量潮）', fontsize=14, fontweight='bold')
ax2.set_xlabel('日期', fontsize=12)
ax2.set_ylabel('OBV', fontsize=12)
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<h3 style="font-family: SimSun, 宋体, serif; font-size: 12pt; line-height: 1.5; margin: 0; text-align: justify;">总结</h3>
<p style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
本任务完成了以下内容：
</p>
<ul style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">
<li>对数据进行了全面的诊断分析，包括缺失值检查、描述性统计、分布分析等</li>
<li>详细介绍了 RSI、MACD、布林带三个经典技术指标的计算方法和作用</li>
<li>使用 Python 编程实现了上述三个指标的计算，并绘制了美观的可视化图形</li>
<li>扩展介绍了 OBV（能量潮）指标，包括其定义、计算方法和作用，并进行了编程实现</li>
</ul>
</div>